# **Install dependencies**

In [ ]:
!pip install -qU "requests>=2.32.5" \
  langchain langchain-community langchain-chroma chromadb \
  pypdf sentence-transformers faiss-cpu tiktoken \
  transformers accelerate

# **Core imports**

In [ ]:
import os, re, json, random, time, requests
from pathlib import Path
from typing import List, Tuple

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma  # no deprecation

# Local HF inference (fallback)
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


# **Configuration**

In [ ]:
# Book & storage paths
BOOK_URL = os.environ.get(
    "BOOK_URL",
    "https://github.com/infoalpha/Data-Science-books/blob/master/storytelling-with-data-cole-nussbaumer-knaflic.pdf"
)
PDF_PATH = os.environ.get("BOOK_PDF", "storytelling-with-data.pdf")
PERSIST_DIR = os.environ.get("CHROMA_DIR", ".chroma_swd")
USE_CHROMA = os.environ.get("USE_CHROMA", "1") != "0"

# Models / embeddings
EMBEDDING_MODEL   = os.environ.get("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
HF_TOKEN          = os.environ.get("HUGGINGFACEHUB_API_TOKEN")
HF_MODEL_ID       = os.environ.get("HF_MODEL_ID", "mistralai/Mistral-7B-Instruct-v0.3")  # API path (may 404)
HF_LOCAL_MODEL_ID = os.environ.get("HF_LOCAL_MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")     # local fallback
GEN_BACKEND       = os.environ.get("GEN_BACKEND", "auto").lower()  # "auto" | "api" | "local"

# RAG parameters
TOP_K            = int(os.environ.get("TOP_K", "6"))
SIM_THRESHOLD    = float(os.environ.get("SIM_THRESHOLD", "0.25"))
CHUNK_SIZE       = int(os.environ.get("CHUNK_SIZE", "900"))
CHUNK_OVERLAP    = int(os.environ.get("CHUNK_OVERLAP", "120"))
MAX_CTX_CHARS    = int(os.environ.get("MAX_CTX_CHARS", "14000"))
MAX_NEW_TOKENS   = int(os.environ.get("MAX_NEW_TOKENS", "512"))
TEMPERATURE      = float(os.environ.get("TEMPERATURE", "0.2"))

# Public Inference API candidates (tried in order when GEN_BACKEND is "api" or "auto")
FALLBACK_API_MODELS = [
    HF_MODEL_ID,
    "HuggingFaceTB/SmolLM3-3B",   # sometimes served
    "Qwen/Qwen2.5-7B-Instruct",   # may need license accept (403)
    "google/flan-t5-xl",          # text2text; often served
]


# **Utilities**

In [ ]:
def to_raw_github(url: str) -> str:
    if "github.com/" in url and "/blob/" in url:
        return url.replace("github.com/", "raw.githubusercontent.com/").replace("/blob/", "/")
    return url

def download_file(url: str, dest: str):
    url = to_raw_github(url)
    print(f"Downloading PDF from: {url}")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

def build_embeddings():
    return HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

def collapse_page_runs(pages: List[int]) -> str:
    if not pages:
        return ""
    pages = sorted(set(int(p) for p in pages if p))
    runs, start = [], pages[0]
    prev = start
    for p in pages[1:]:
        if p == prev + 1:
            prev = p
        else:
            runs.append((start, prev))
            start = prev = p
    runs.append((start, prev))
    parts = [f"{a}" if a == b else f"{a}–{b}" for (a, b) in runs]
    return ", ".join(parts)

def citations_from_docs(docs: List[Document]) -> str:
    pages = [int(d.metadata.get("page", 0)) for d in docs if "page" in d.metadata]
    if not pages:
        return ""
    runs = collapse_page_runs(pages)
    return f"pp. {runs}" if ("," in runs or "–" in runs) else f"p. {runs}"

def trim_context_to_chars(chunks: List[str], max_chars: int) -> str:
    out, total = [], 0
    for c in chunks:
        c = c.strip()
        if not c:
            continue
        if total + len(c) + 2 > max_chars:
            break
        out.append(c)
        total += len(c) + 2
    return "\n\n".join(out)


# **Generator backend**

In [ ]:
class GeneratorBackend:
    def __init__(self):
        self.mode = None             # "api" or "local"
        self.api_model_id = None
        self.local_pipe = None
        self._init_backend()

    # ------- API path -------
    def _api_available(self, model_id: str) -> bool:
        if not HF_TOKEN:
            return False
        try:
            r = requests.get(
                f"https://api-inference.huggingface.co/models/{model_id}",
                headers={"Authorization": f"Bearer {HF_TOKEN}"}, timeout=20
            )
            return r.status_code == 200
        except Exception:
            return False

    def _init_api(self) -> bool:
        for mid in [m for m in FALLBACK_API_MODELS if m]:
            if self._api_available(mid):
                self.mode = "api"
                self.api_model_id = mid
                print(f"[Gen] Using HF Inference API model: {mid}")
                return True
            else:
                print(f"[Gen] {mid} not available on public Inference API; trying next...")
        return False

    # ------- Local path -------
    def _init_local(self) -> bool:
        model_id = HF_LOCAL_MODEL_ID
        print(f"[Gen] Loading local Transformers model: {model_id} (this can take a minute the first time)")
        tok = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True)
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",          # GPU if available; CPU otherwise
            torch_dtype="auto",
            trust_remote_code=True
        )
        self.local_pipe = pipeline(
            "text-generation",
            model=mdl,
            tokenizer=tok,
        )
        self.mode = "local"
        return True

    # ------- Init selection -------
    def _init_backend(self):
        if GEN_BACKEND == "api":
            if not self._init_api():
                raise RuntimeError("GEN_BACKEND=api but no public HF Inference model is available for your account.")
            return
        if GEN_BACKEND == "local":
            self._init_local()
            return
        # auto: try API first, else local
        if not self._init_api():
            self._init_local()

    # ------- Generate -------
    def generate(self, prompt: str, max_new_tokens: int, temperature: float) -> str:
        if self.mode == "api":
            headers = {"Authorization": f"Bearer {HF_TOKEN}"}
            url = f"https://api-inference.huggingface.co/models/{self.api_model_id}"
            payload = {
                "inputs": prompt,
                "parameters": {
                    "temperature": temperature,
                    "max_new_tokens": max_new_tokens,
                    "return_full_text": False,
                    "do_sample": temperature > 0
                }
            }
            for attempt in range(4):
                r = requests.post(url, headers=headers, json=payload, timeout=180)
                if r.status_code in (404, 403):
                    # flip to local immediately
                    print(f"[Gen] API {self.api_model_id} returned {r.status_code}; switching to local backend.")
                    self._init_local()
                    break
                if r.status_code in (429, 500, 502, 503, 504):
                    time.sleep(1.5 * (attempt + 1))
                    continue
                r.raise_for_status()
                data = r.json()
                if isinstance(data, list) and data and "generated_text" in data[0]:
                    return data[0]["generated_text"].strip()
                if isinstance(data, dict) and "generated_text" in data:
                    return data["generated_text"].strip()
                return json.dumps(data)[:2000]
            # fell through due to 404/403: local will handle below

        # Local path
        outs = self.local_pipe(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(0.0, temperature),
            return_full_text=False
        )
        text = outs[0]["generated_text"]
        return text.strip()


# **TutorRAG (RAG core)**

In [ ]:
class TutorRAG:
    def __init__(self):
        if not Path(PDF_PATH).exists():
            download_file(BOOK_URL, PDF_PATH)
        self.embeddings = build_embeddings()
        self.vs = self._load_or_build_index()
        self.chat_history: List[Tuple[str, str]] = []
        self.gen = GeneratorBackend()

    def _load_or_build_index(self):
        if USE_CHROMA and Path(PERSIST_DIR).exists() and any(Path(PERSIST_DIR).iterdir()):
            return Chroma(persist_directory=PERSIST_DIR, embedding_function=self.embeddings)
        docs = self._load_and_chunk(PDF_PATH)
        if USE_CHROMA:
            vs = Chroma.from_documents(documents=docs, embedding=self.embeddings, persist_directory=PERSIST_DIR)
            vs.persist()
            return vs
        else:
            from langchain_community.vectorstores import FAISS
            return FAISS.from_documents(documents=docs, embedding=self.embeddings)

    def _load_and_chunk(self, pdf_path: str) -> List[Document]:
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()
        for d in pages:
            d.metadata["page"] = int(d.metadata.get("page", 0)) + 1
            d.metadata["source"] = pdf_path
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", "? ", "! ", "; ", ": ", ", ", " ", ""]
        )
        chunks: List[Document] = []
        for d in pages:
            for c in splitter.split_text(d.page_content):
                chunks.append(Document(page_content=c, metadata=d.metadata.copy()))
        return chunks

    def retrieve(self, query: str, k: int = TOP_K) -> Tuple[List[Document], List[float]]:
        docs, scores = [], []
        try:
            results = self.vs.similarity_search_with_relevance_scores(query, k=k)
            if results and isinstance(results[0], tuple):
                docs = [d for d, s in results]
                scores = [float(s) for _, s in results]
        except Exception:
            docs = self.vs.similarity_search(query, k=k)
            scores = [1.0] * len(docs)
        return docs, scores

    def _build_prompt(self, question: str, context_blocks: List[str]) -> str:
        history_text = ""
        if self.chat_history:
            recent = self.chat_history[-4:]
            pairs = []
            for u, a in recent:
                pairs.append(f"User: {u}")
                pairs.append(f"Tutor: {a}")
            history_text = "\n".join(pairs)

        context_text = trim_context_to_chars(context_blocks, MAX_CTX_CHARS)

        system_rules = (
            "You are TutorAI, a strict tutor that ONLY answers using the provided book excerpts.\n"
            "- If the answer is not contained in the excerpts, say: "
            "\"I can't answer that from *Storytelling with Data*; it's not in the provided text.\"\n"
            "- NEVER use outside knowledge. NEVER hallucinate chart definitions or terms not present.\n"
            "- Be concise, clear, and pedagogical. Use bullets when helpful.\n"
            "- Do not invent citations; only refer to the pages in the excerpts (page numbers are provided).\n"
        )

        return f"""
{system_rules}

Conversation (recent):
{history_text}

Question:
{question}

Book excerpts (each block shows [page n] and text):
{context_text}

Instructions:
- Answer ONLY using the excerpts above.
- If insufficient information is present, refuse as instructed.
- Keep the answer well-structured for a learner.
- Do NOT add sources beyond the book.
""".strip()

    def answer(self, question: str) -> str:
        docs, scores = self.retrieve(question, k=TOP_K)
        best = max(scores) if scores else 0.0
        if best < SIM_THRESHOLD or not docs:
            refusal = "I can't answer that from *Storytelling with Data*; it's not in the provided text."
            self.chat_history.append((question, refusal))
            return refusal

        ctx_blocks = [f"[page {int(d.metadata.get('page',0))}]\n{d.page_content}" for d in docs]
        prompt = self._build_prompt(question, ctx_blocks)
        raw = self.gen.generate(prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE)
        cites = citations_from_docs(docs)
        answer = raw.strip()
        if cites:
            answer = f"{answer}\n\nCitations: {cites}"
        self.chat_history.append((question, answer))
        return answer

    def summarize(self, topic_hint: str, sentences: int = 5) -> str:
        return self.answer(f"Summarize the following topic in about {sentences} sentences: {topic_hint}")

    def explain(self, concept: str) -> str:
        return self.answer(f"Explain this concept to a learner with examples from the book: {concept}")

    def quiz(self, n: int = 20) -> List[str]:
        questions: List[str] = []
        seeds = [
            "table of contents", "introduction", "story", "visual clutter", "bar chart", "line chart",
            "color", "layout", "audience", "annotations", "axis", "title", "iteration", "storytelling", "case study"
        ]
        random.shuffle(seeds)

        def questions_from_chunk(doc: Document) -> List[str]:
            pg = int(doc.metadata.get("page", 0))
            prompt = f"""
You will write concise quiz questions ONLY based on this book excerpt.

[page {pg}]
{doc.page_content}

Write 2 quiz questions (no answers). Each question MUST be answerable from the excerpt itself.
Do not use outside knowledge. Return as:
1) <question> (p. {pg})
2) <question> (p. {pg})
"""
            out = self.gen.generate(prompt, max_new_tokens=220, temperature=0.2)
            lines = [re.sub(r"\s+", " ", L).strip() for L in out.splitlines() if L.strip()]
            qlines = []
            for L in lines:
                m = re.search(r"^\d+\)\s*(.+)$", L)
                if m:
                    qlines.append(m.group(1).strip())
            if not qlines:
                parts = [p.strip()+"?" for p in out.split("?") if p.strip()]
                qlines = parts[:2]
            qlines = [q if "(p." in q else f"{q} (p. {pg})" for q in qlines]
            return qlines[:2]

        seen_pages = set()
        for seed in seeds:
            docs, _ = self.retrieve(seed, k=1)
            if not docs:
                continue
            d = docs[0]
            pg = int(d.metadata.get("page", 0))
            if pg in seen_pages:
                continue
            seen_pages.add(pg)
            for q in questions_from_chunk(d):
                q = q.strip()
                if q and not q.endswith("?"):
                    q = q.rstrip(".") + "?"
                questions.append(q)
                if len(questions) >= n:
                    break
            if len(questions) >= n:
                break

        while len(questions) < n:
            docs, _ = self.retrieve("data visualization", k=1)
            if not docs:
                break
            for q in questions_from_chunk(docs[0]):
                q = q.strip()
                if q and q not in questions:
                    if not q.endswith("?"):
                        q = q.rstrip(".") + "?"
                    questions.append(q)
                if len(questions) >= n:
                    break

        return questions[:n]


# **Demo (5 interactions)**

In [ ]:
if __name__ == "__main__":
    rag = TutorRAG()
    demo_questions = [
        "What are some common mistakes people make when designing charts and graphs?",
        "What is a slopegraph and when should it be used?",
        "Prepare me a quiz of 20 questions from the book to test my knowledge.",
        "Explain how to reduce clutter in a chart and why it matters.",
        "Summarize the main message of the book’s introduction."
    ]
    for i, q in enumerate(demo_questions, 1):
        print("="*90)
        print(f"Q{i}: {q}")
        if "quiz" in q.lower():
            quiz = rag.quiz(n=20)
            print("Tutor:", "\n- " + "\n- ".join(quiz))
            print("Note: Each question includes page citation tags.")
        else:
            a = rag.answer(q)
            print("Tutor:", a)


[Gen] mistralai/Mistral-7B-Instruct-v0.3 not available on public Inference API; trying next...
[Gen] HuggingFaceTB/SmolLM3-3B not available on public Inference API; trying next...
[Gen] Qwen/Qwen2.5-7B-Instruct not available on public Inference API; trying next...
[Gen] google/flan-t5-xl not available on public Inference API; trying next...
[Gen] Loading local Transformers model: Qwen/Qwen2.5-0.5B-Instruct (this can take a minute the first time)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cpu


Q1: What are some common mistakes people make when designing charts and graphs?
Tutor: - Distractions, eliminating,  
132–135
- Duarte, Nancy, 22, 30, 72, 173 
- 179
- Effective visuals, choosing, 14, 
35–69
- graphs, 43–49
- area graphs, 59–60
- bar charts, 50–59
- lines, 45–49
- points, 44–45
- slopegraph, 47–49
- infographics, 60–61
- simple text, 38–40
- tables, 40–43
- borders, 41
- heatmap, 42–43
- visual aids, 112
- Enclosure principle, 77
- Excel, 13, 42, 244
- changing components of a 
graph in, 196
- slopegraph template, 48
- Exploratory vs. explanatory 
analysis, 19–20,  
112
- Where to go from here 249
- You now have a discerning eye when it comes to the visual display 
of information. You will never look at a graph the same. One work-
shop attendee told me that he is “ruined”—he can’t encounter a 
data visualization without applying his new lens for assessing effec-
tiveness. I love hearing these stories, as it means I’m making progress 
toward my goal of ridding the world

KeyboardInterrupt: 

# **Install Gradio**

In [ ]:
!pip install -qU gradio

# **Gradio Chat UI**

In [ ]:
import gradio as gr
import traceback
from pathlib import Path

# One global TutorRAG instance for the app
APP_RAG = None

def ui_init():
    """Create/refresh the global TutorRAG instance."""
    global APP_RAG
    try:
        APP_RAG = TutorRAG()
        msg = "✅ Ready. Book is indexed." if Path(PDF_PATH).exists() else "✅ Ready."
        return msg, []  # status text, empty chat
    except Exception as e:
        tb = traceback.format_exc()
        return f"❌ Init failed: {e}\n\n```\n{tb}\n```", []

def ui_set_params(top_k, sim_thr):
    """Update global knobs used by TutorRAG.answer()."""
    try:
        global TOP_K, SIM_THRESHOLD
        TOP_K = int(top_k)
        SIM_THRESHOLD = float(sim_thr)
        return f"ℹ️ Params set → top_k={TOP_K}, sim_threshold={SIM_THRESHOLD:.2f}"
    except Exception as e:
        return f"❌ Could not set params: {e}"

def ui_ingest_pdf(file):
    """Optional: swap to a different PDF and rebuild the index."""
    global APP_RAG
    if file is None:
        return "⚠️ No file uploaded."
    if APP_RAG is None:
        return "⚠️ Not initialized. Click **Init / Reload** first."
    try:
        pdf_path = file.name
        docs = APP_RAG._load_and_chunk(pdf_path)
        if USE_CHROMA:
            tmp_dir = f"{PERSIST_DIR}_upload"
            vs = Chroma.from_documents(documents=docs, embedding=APP_RAG.embeddings, persist_directory=tmp_dir)
            vs.persist()
            APP_RAG.vs = vs
        else:
            from langchain_community.vectorstores import FAISS
            APP_RAG.vs = FAISS.from_documents(documents=docs, embedding=APP_RAG.embeddings)
        APP_RAG.chat_history = []
        return f"✅ Ingested **{Path(pdf_path).name}** — {len(docs)} chunks."
    except Exception as e:
        tb = traceback.format_exc()
        return f"❌ Upload/ingest failed: {e}\n\n```\n{tb}\n```"

def ui_ask(message, chat_history, top_k, sim_thr):
    """Main chat handler."""
    global APP_RAG
    try:
        chat_history = chat_history or []
        if not message or not message.strip():
            return "", chat_history, "⚠️ Please type a question."
        if APP_RAG is None:
            chat_history.append((message, "⚠️ Not initialized. Click **Init / Reload** first."))
            return "", chat_history, "⚠️ RAG not initialized."

        # Apply per-turn params
        status = ui_set_params(top_k, sim_thr)

        # Grounded answer
        answer = APP_RAG.answer(message)
        chat_history.append((message, answer))
        return "", chat_history, status or "✅ Answered."
    except Exception as e:
        tb = traceback.format_exc()
        chat_history.append((message, f"❌ Error: {e}"))
        return "", chat_history, f"❌ Exception:\n\n```\n{tb}\n```"

def ui_clear():
    global APP_RAG
    try:
        if APP_RAG is not None:
            APP_RAG.chat_history = []
        return [], "🧹 Cleared."
    except Exception as e:
        tb = traceback.format_exc()
        return [], f"❌ Could not clear: {e}\n\n```\n{tb}\n```"

with gr.Blocks(css="#chatbot {height: 520px}") as demo:
    gr.Markdown("## TutorAI (RAG over *Storytelling with Data*) — Chat")

    with gr.Row():
        init_btn = gr.Button("Init / Reload", variant="primary")
        status = gr.Markdown("")  # live status / errors

    with gr.Row():
        file_up = gr.File(
            label="Optional: Upload a different PDF",
            file_count="single",
            file_types=[".pdf"]
        )
        top_k = gr.Slider(2, 12, value=TOP_K, step=1, label="Top-K passages")
        sim_thr = gr.Slider(0.0, 1.0, value=SIM_THRESHOLD, step=0.05, label="Similarity threshold")

    with gr.Row():
        chat = gr.Chatbot(label="TutorAI", elem_id="chatbot")

    with gr.Row():
        msg = gr.Textbox(placeholder="Ask from the book… e.g., 'What are common chart design mistakes?'")
    with gr.Row():
        send = gr.Button("Send", variant="primary")
        clear = gr.Button("Clear")

    # Wire events (no gr.State involved)
    init_btn.click(fn=ui_init, outputs=[status, chat])
    file_up.upload(fn=ui_ingest_pdf, inputs=[file_up], outputs=[status])
    send.click(fn=ui_ask, inputs=[msg, chat, top_k, sim_thr], outputs=[msg, chat, status])
    msg.submit(fn=ui_ask, inputs=[msg, chat, top_k, sim_thr], outputs=[msg, chat, status])
    clear.click(fn=ui_clear, outputs=[chat, status])

# Show tracebacks in the page (super helpful in Colab)
demo.launch(share=True, debug=True)


/tmp/ipython-input-3319885581.py:103: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chat = gr.Chatbot(label="TutorAI", elem_id="chatbot")


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://411a73ce9125b9b81a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipython-input-2858083266.py:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cpu


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://411a73ce9125b9b81a.gradio.live
